# Betonfestigkeit vorhersagen: Random Forest vs. TabPFN

Ein Betonwerk kennt nur wenige geprüfte Mischungen. Kann ein Modell daraus die **Druckfestigkeit (in MPa)** neuer Mischungen vorhersagen?

Datensatz: 1.030 Betonmischungen (OpenML) – Zutaten pro m³, Alter in Tagen, gemessene Festigkeit.

In [ ]:
!pip install -q tabpfn-client

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from tabpfn_client import TabPFNRegressor, set_access_token

set_access_token("HIER_TOKEN_EINTRAGEN")  # Token von https://ux.priorlabs.ai

## 1. Daten laden

In [ ]:
daten = fetch_openml(data_id=44959, as_frame=True, parser="auto").frame
X = daten.drop(columns="strength")
y = daten["strength"]

# 500 Mischungen beiseitelegen: daran prüfen wir die Vorhersagen
X_test, X_rest, y_test, y_rest = train_test_split(X, y, train_size=500, random_state=42)

daten.head(10)

## 2. Beide Modelle mit nur 50 bekannten Mischungen

In [ ]:
X_klein, _, y_klein, _ = train_test_split(X_rest, y_rest, train_size=50, random_state=42)

rf = RandomForestRegressor(random_state=42).fit(X_klein, y_klein)
tabpfn = TabPFNRegressor().fit(X_klein, y_klein)

vorhersage_rf = rf.predict(X_test)
vorhersage_tabpfn = tabpfn.predict(X_test)

print(f"Random Forest: im Schnitt {mean_absolute_error(y_test, vorhersage_rf):.1f} MPa daneben")
print(f"TabPFN:        im Schnitt {mean_absolute_error(y_test, vorhersage_tabpfn):.1f} MPa daneben")

## 3. Diagramm: Vorhersage vs. Wirklichkeit

In [ ]:
fig, achsen = plt.subplots(1, 2, figsize=(11, 4.8), sharey=True)
for ax, name, vorhersage, farbe in [(achsen[0], "Random Forest", vorhersage_rf, "#E07B39"),
                                    (achsen[1], "TabPFN", vorhersage_tabpfn, "#2E7D5B")]:
    ax.scatter(y_test, vorhersage, s=10, alpha=0.5, color=farbe)
    ax.plot([0, 85], [0, 85], color="gray", linestyle="--")   # perfekte Vorhersage
    ax.set_title(f"{name}: im Schnitt {mean_absolute_error(y_test, vorhersage):.1f} MPa daneben", loc="left")
    ax.set_xlabel("Tatsächliche Festigkeit (MPa)")
achsen[0].set_ylabel("Vorhergesagte Festigkeit (MPa)")
plt.tight_layout()
plt.show()

## 4. Zufall oder echter Unterschied?

Wir wiederholen den Vergleich für verschiedene Datenmengen je 10-mal mit anderen Zufallsstichproben.
Beide Modelle bekommen jeweils **genau dieselben Daten**. Der Wilcoxon-Test prüft, ob TabPFN systematisch besser ist (p < 0,05 = kein Zufall).

⏱️ Dauert einige Minuten (50 TabPFN-Aufrufe).

In [ ]:
ergebnisse = []
for anzahl in [30, 50, 100, 200, 500]:
    for wiederholung in range(10):
        X_tr, _, y_tr, _ = train_test_split(X_rest, y_rest, train_size=anzahl, random_state=wiederholung)
        fehler_rf = mean_absolute_error(y_test, RandomForestRegressor(random_state=0).fit(X_tr, y_tr).predict(X_test))
        fehler_tabpfn = mean_absolute_error(y_test, TabPFNRegressor().fit(X_tr, y_tr).predict(X_test))
        ergebnisse.append({"Mischungen": anzahl, "Random Forest": fehler_rf, "TabPFN": fehler_tabpfn})
    print(anzahl, "fertig")
ergebnisse = pd.DataFrame(ergebnisse)

tabelle = ergebnisse.groupby("Mischungen").mean()
tabelle["p-Wert"] = ergebnisse.groupby("Mischungen").apply(lambda g: wilcoxon(g["Random Forest"], g["TabPFN"]).pvalue)
tabelle.round(3)

## 5. Diagramm: Fehler je nach Datenmenge

In [ ]:
mittel = ergebnisse.groupby("Mischungen").mean()
streuung = ergebnisse.groupby("Mischungen").std()

fig, ax = plt.subplots(figsize=(9, 4.8))
for name, farbe in [("Random Forest", "#E07B39"), ("TabPFN", "#2E7D5B")]:
    ax.errorbar(mittel.index, mittel[name], yerr=streuung[name], marker="o", capsize=3, label=name, color=farbe)
ax.set_xscale("log")
ax.set_xticks(mittel.index, mittel.index)
ax.minorticks_off()
ax.set_xlabel("Anzahl bekannter Mischungen")
ax.set_ylabel("Mittlerer Fehler (MPa) – kleiner = besser")
ax.set_title("Betonfestigkeit: Random Forest vs. TabPFN (10 Wiederholungen)", loc="left")
ax.legend()
plt.tight_layout()
plt.show()